# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


My rule (plain words): I score every page-day by how far its actual CTR falls below the expected CTR for its position tier (top_10 vs beyond_10, from ML-07's earlier signal confirmation). A bigger gap means the page is underperforming more than its ranking position would predict — that page gets prioritized higher in the review queue.

Reason codes this rule can output:
- CTR_BELOW_EXPECTED_FOR_POSITION: the page's CTR is meaningfully below what its position tier predicts.
- WITHIN_EXPECTED_RANGE: the page's CTR is close to or above what its position predicts — no action needed.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import pandas as pd
import os
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

rule_df = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks * 1.0 / gsc_impressions as ctr,
    gsc_sum_position * 1.0 / gsc_impressions as avg_position
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
""").df()

rule_df['expected_ctr'] = rule_df['avg_position'].apply(lambda p: 0.0039 if p <= 10 else 0.0018)
rule_df['score'] = rule_df['expected_ctr'] - rule_df['ctr']
rule_df['reason_code'] = rule_df['score'].apply(
    lambda s: 'CTR_BELOW_EXPECTED_FOR_POSITION' if s > 0.001 else 'WITHIN_EXPECTED_RANGE'
)
rule_df['action'] = rule_df['reason_code'].apply(
    lambda r: 'REVIEW_TITLE_META' if r == 'CTR_BELOW_EXPECTED_FOR_POSITION' else 'MONITOR'
)

# minimum impression floor, addressing the weakness found in the earlier top-10 review
rule_df_filtered = rule_df[rule_df['gsc_impressions'] >= 50].copy()

ranked_queue = rule_df_filtered.sort_values('score', ascending=False).reset_index(drop=True)

os.makedirs('work/outputs', exist_ok=True)
ranked_queue.to_csv('work/outputs/baseline_action_score.csv', index=False)

ranked_queue.head(20)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,gsc_impressions,ctr,avg_position,expected_ctr,score,reason_code,action
0,client_20259bd6705d81d4,content_3e662420c59ca74e,104,0.0,4.019231,0.0039,0.0039,CTR_BELOW_EXPECTED_FOR_POSITION,REVIEW_TITLE_META
1,client_20259bd6705d81d4,content_db0c67d6d4cdb594,57,0.0,7.508772,0.0039,0.0039,CTR_BELOW_EXPECTED_FOR_POSITION,REVIEW_TITLE_META
2,client_20259bd6705d81d4,content_568a352a39556f3e,54,0.0,5.703704,0.0039,0.0039,CTR_BELOW_EXPECTED_FOR_POSITION,REVIEW_TITLE_META
3,client_20259bd6705d81d4,content_b348bfd011489cea,70,0.0,2.700000,0.0039,0.0039,CTR_BELOW_EXPECTED_FOR_POSITION,REVIEW_TITLE_META
4,client_73cda7b4e4f265ea,content_a6ae52d901485985,177,0.0,6.531073,0.0039,0.0039,CTR_BELOW_EXPECTED_FOR_POSITION,REVIEW_TITLE_META
5,client_73cda7b4e4f265ea,content_538d2f2226126655,156,0.0,5.358974,0.0039,0.0039,CTR_BELOW_EXPECTED_FOR_POSITION,REVIEW_TITLE_META
6,client_73cda7b4e4f265ea,content_3e966751b7ad947b,65,0.0,6.230769,0.0039,0.0039,CTR_BELOW_EXPECTED_FOR_POSITION,REVIEW_TITLE_META
7,client_73cda7b4e4f265ea,content_0d510f7a6abb761e,54,0.0,6.537037,0.0039,0.0039,CTR_BELOW_EXPECTED_FOR_POSITION,REVIEW_TITLE_META
8,client_73cda7b4e4f265ea,content_7943308a4f7c328f,59,0.0,7.864407,0.0039,0.0039,CTR_BELOW_EXPECTED_FOR_POSITION,REVIEW_TITLE_META
9,client_73cda7b4e4f265ea,content_22b8b1c287f3cf9c,53,0.0,8.811321,0.0039,0.0039,CTR_BELOW_EXPECTED_FOR_POSITION,REVIEW_TITLE_META


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Top-20 review:

All 20 rows share reason_code CTR_BELOW_EXPECTED_FOR_POSITION and the same score (0.0039) — every one is a top-10-position page with 0 clicks. Confidence generally scales with impressions: rows with 200+ impressions (e.g., rows 14, 17, 19 at 215, 214, 531 impressions) are high-confidence — 0 clicks on hundreds of impressions is a real, trustworthy signal. Rows with 50-70 impressions (e.g., rows 2, 3, 7, 8) are moderate-confidence — still meaningful, but a few more impressions could shift the picture.

Row 0 (104 impressions, position 4.0): Action REVIEW_TITLE_META. Why: strong position, zero clicks, decent volume. What would make it wrong: if this page recently changed URLs or had a title update mid-month, the 0-click count could reflect a transition period, not a true title/meta problem.

Row 19 (531 impressions, position 0.80 — essentially position #1): Action REVIEW_TITLE_META. Why: this is the single strongest case in the list — ranking #1 on average with 531 impressions and 0 clicks is a major red flag, not noise. What would make it wrong: extremely unlikely to be wrong given the volume; if anything this deserves top priority, not just top-20 inclusion.

Row 15 & 16 (79 and 70 impressions, both position ~1.8): Action REVIEW_TITLE_META. Why: near-top position, meaningful volume, zero clicks. What would make it wrong:

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Weak picks:
The biggest weakness isn't any individual row — it's the client concentration. 15 of the top 20 rows come from one client (client_73cda7b4e4f265ea). This could mean: (a) this client genuinely has many underperforming pages, or (b) this client has a data/tracking anomaly affecting all their pages uniformly (e.g., a broken conversion pixel or a GSC sync issue), which would make all 15 rows "wrong" for the same underlying reason rather than 15 independent findings. This needs a manual check with the actual client context before acting on it as 15 separate action items.

Leakage check: I confirm no product flags or future-window data were used in this rule. All inputs (gsc_impressions, gsc_clicks, gsc_sum_position) are same-day, point-in-time GSC metrics — none of them reference future dates, no client-provided "confirmed issue" flags were used, and the label (score) is computed only from the same day's ctr and avg_position, consistent with the point-in-time features validated in ML-05's leakage hunt.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.